### Question

Build a gradio UI where a user can supply a textual input and the output is the sentiment of that input.

### Instructions


1. figure out how to generate sentiment using a hf model - dont worry about gradio yet. you are free to use any hf model of your choice.
3. once summarization is figured, build the gradio UI using Interface!

In [1]:
"""
Sentiment Analysis App using Gradio + HuggingFace Transformers

Setup:
    pip install gradio transformers torch

Run:
    python sentiment_app.py
"""

import gradio as gr
from transformers import pipeline

# ── Load model once at startup ──────────────────────────────────────────────
print("Loading sentiment model…")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)
print("Model ready.")

# Emoji map for labels
EMOJI = {"POSITIVE": "😊", "NEGATIVE": "😞", "NEUTRAL": "😐"}
COLOR = {"POSITIVE": "#22c55e", "NEGATIVE": "#ef4444", "NEUTRAL": "#f59e0b"}


def analyse(text: str) -> tuple[str, str]:
    """Return (label_with_emoji, confidence_bar_html)."""
    if not text.strip():
        return "—", "<p style='color:#888'>Enter some text above to begin.</p>"

    result = sentiment_pipeline(text[:512])[0]   # cap at 512 tokens
    label: str  = result["label"]                # e.g. "POSITIVE"
    score: float = result["score"]               # e.g. 0.9987

    emoji  = EMOJI.get(label, "🔍")
    color  = COLOR.get(label, "#6366f1")
    pct    = round(score * 100, 1)

    sentiment_out = f"{emoji}  {label.capitalize()}"

    confidence_html = f"""
    <div style="
        font-family: 'DM Mono', monospace;
        background: #0f0f0f;
        border: 1px solid #222;
        border-radius: 12px;
        padding: 20px 24px;
        margin-top: 4px;
    ">
        <div style="
            display: flex;
            justify-content: space-between;
            align-items: baseline;
            margin-bottom: 10px;
        ">
            <span style="color:#aaa; font-size:0.8rem; letter-spacing:0.08em; text-transform:uppercase;">
                Confidence
            </span>
            <span style="color:{color}; font-size:1.4rem; font-weight:700;">
                {pct}%
            </span>
        </div>
        <div style="
            background: #1e1e1e;
            border-radius: 999px;
            height: 10px;
            overflow: hidden;
        ">
            <div style="
                width: {pct}%;
                height: 100%;
                background: linear-gradient(90deg, {color}88, {color});
                border-radius: 999px;
                transition: width 0.5s ease;
            "></div>
        </div>
    </div>
    """
    return sentiment_out, confidence_html


# ── Custom CSS ───────────────────────────────────────────────────────────────
css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Syne:wght@600;700;800&display=swap');

body, .gradio-container {
    background: #080808 !important;
    font-family: 'DM Mono', monospace !important;
    color: #e5e5e5 !important;
}

.gradio-container {
    max-width: 700px !important;
    margin: 0 auto !important;
}

/* Header */
#header {
    text-align: center;
    padding: 48px 0 32px;
}
#header h1 {
    font-family: 'Syne', sans-serif !important;
    font-size: 2.6rem;
    font-weight: 800;
    letter-spacing: -0.03em;
    background: linear-gradient(135deg, #e5e5e5 0%, #666 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin: 0 0 8px;
}
#header p {
    color: #555;
    font-size: 0.85rem;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    margin: 0;
}

/* Inputs / Outputs */
label > span {
    font-size: 0.7rem !important;
    letter-spacing: 0.1em !important;
    text-transform: uppercase !important;
    color: #555 !important;
}

textarea, input[type="text"] {
    background: #0f0f0f !important;
    border: 1px solid #222 !important;
    border-radius: 12px !important;
    color: #e5e5e5 !important;
    font-family: 'DM Mono', monospace !important;
    font-size: 0.95rem !important;
    caret-color: #e5e5e5;
    padding: 14px 16px !important;
    transition: border-color 0.2s;
}
textarea:focus {
    border-color: #444 !important;
    outline: none !important;
}

/* Button */
button.primary {
    background: #e5e5e5 !important;
    color: #080808 !important;
    font-family: 'Syne', sans-serif !important;
    font-weight: 700 !important;
    font-size: 0.85rem !important;
    letter-spacing: 0.08em !important;
    text-transform: uppercase !important;
    border: none !important;
    border-radius: 10px !important;
    padding: 12px 28px !important;
    cursor: pointer !important;
    transition: opacity 0.2s !important;
}
button.primary:hover { opacity: 0.85 !important; }

/* Sentiment label output */
#sentiment-out textarea {
    font-family: 'Syne', sans-serif !important;
    font-size: 1.6rem !important;
    font-weight: 700 !important;
    text-align: center !important;
    color: #e5e5e5 !important;
    padding: 20px !important;
}

/* Examples section */
.examples-header { color: #444 !important; }
table.examples {
    border-collapse: collapse;
}
table.examples td {
    background: #0f0f0f !important;
    border: 1px solid #1a1a1a !important;
    color: #888 !important;
    font-size: 0.82rem !important;
    padding: 8px 14px !important;
    cursor: pointer;
    transition: background 0.15s;
}
table.examples tr:hover td { background: #161616 !important; color: #ccc !important; }

/* Footer */
#footer {
    text-align: center;
    padding: 32px 0 48px;
    color: #333;
    font-size: 0.72rem;
    letter-spacing: 0.08em;
}
"""

# ── Build UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(css=css, title="Sentiment Analysis") as demo:

    gr.HTML("""
    <div id="header">
        <h1>Sentiment Analysis</h1>
        <p>Powered by HuggingFace · DistilBERT · SST-2</p>
    </div>
    """)

    with gr.Column():
        text_input = gr.Textbox(
            label="Input Text",
            placeholder="Type or paste any text here…",
            lines=4,
            max_lines=8,
        )

        analyse_btn = gr.Button("Analyse Sentiment", variant="primary")

        sentiment_out = gr.Textbox(
            label="Sentiment",
            interactive=False,
            elem_id="sentiment-out",
        )

        confidence_out = gr.HTML(
            value="<p style='color:#333; font-size:0.8rem; text-align:center; padding:12px 0'>Confidence score will appear here.</p>"
        )

    gr.Examples(
        examples=[
            ["I absolutely loved this product — it exceeded all my expectations!"],
            ["The service was terrible and the staff were rude and unhelpful."],
            ["The weather today is neither good nor bad, just ordinary."],
            ["This film was a masterpiece — the cinematography took my breath away."],
            ["I can't believe how disappointing the whole experience was."],
        ],
        inputs=text_input,
        label="Try an example",
    )

    gr.HTML('<div id="footer">Model: distilbert-base-uncased-finetuned-sst-2-english</div>')

    # Wire up interactions
    analyse_btn.click(
        fn=analyse,
        inputs=text_input,
        outputs=[sentiment_out, confidence_out],
    )
    text_input.submit(
        fn=analyse,
        inputs=text_input,
        outputs=[sentiment_out, confidence_out],
    )

if __name__ == "__main__":
    demo.launch(share=False)

Loading sentiment model…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

Model ready.


/tmp/ipykernel_178/159265123.py:199: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="Sentiment Analysis") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [2]:
import gradio as gr

def analyze_sentiment(text):
    if not text.strip():
        return "Please enter some text!"

    result = sentiment_pipeline(text)[0]

    label = result['label']
    score = result['score'] * 100  # convert to percentage

    # Make the output friendly and readable
    emoji = "😊" if label == "POSITIVE" else "😞"

    output = f"{emoji} Sentiment: {label}\n📊 Confidence: {score:.2f}%"
    return output

# Build the Gradio Interface
demo = gr.Interface(
    fn=analyze_sentiment,                          # function to call
    inputs=gr.Textbox(
        lines=4,
        placeholder="Type something here... e.g. I love this!",
        label="Your Text"
    ),
    outputs=gr.Textbox(label="Sentiment Result"),
    title="💬 Sentiment Analyser",
    description="Type any sentence and find out if it's Positive or Negative!",
    examples=[
        ["I absolutely love this product, it changed my life!"],
        ["This is the worst experience I have ever had."],
        ["Today was just an okay kind of day, nothing special."]
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://66de938996f07f623b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
